In [ ]:
!pip install -q accelerate

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional, Tuple, List

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


---
# 1) Multi-head Latent Attention (MLA)

![](https://moonlight-paper-snapshot.s3.ap-northeast-2.amazonaws.com/arxiv/hardware-centric-analysis-of-deepseeks-multi-head-latent-attention-0.png)

Standard Multi-Head Attention (MHA) caches **full** K and V tensors per token:

```
Cache per token = 2 × num_heads × head_dim  (one for K, one for V)
```

Grouped-Query Attention (GQA, as in LLaMA-2) reduces this by sharing KV heads,
but MLA takes a fundamentally different approach: **low-rank joint compression**.

### MLA key idea

Instead of caching the full K and V matrices, MLA:
1. Projects the hidden state into a *small* compressed latent vector `c_kv`.
2. Reconstructs K and V on-the-fly from `c_kv` during attention.
3. Uses a **decoupled RoPE** branch because RoPE is position-dependent and
   cannot be absorbed into the low-rank compression.

```
Cache per token (MLA) = kv_compress_dim + rope_head_dim
```

For DeepSeek-V2-236B: 512 (compressed) vs 12,288 (full MHA) — **a 24× reduction**.

### Sample Input / Output

```
Input  x:  (batch_num, seq_len, model_dim)   e.g. (2, 64, 2048)
Output o:  (batch_num, seq_len, model_dim)   e.g. (2, 64, 2048)
Cached:    c_kv (batch_num, seq_len, kv_compress_dim)  +
           k_rope (batch_num, seq_len, rope_head_dim)
```

## 1.1 RoPE Helper

![](https://towardsdatascience.com/wp-content/uploads/2024/05/1qj9tbVNQ5pXvpoS_pSx_hQ-1.png)

MLA requires a modified RoPE that operates on arbitrary last-dimension sizes
(not tied to `head_dim`)

In [ ]:
def precompute_rope_frequencies(
    dim: int, max_len: int, theta: float = 10000.0, device: str = "cpu"
) -> torch.Tensor:
    """Precompute complex-valued RoPE frequency tensor.

    Returns:
        freqs_complex: (max_len, dim // 2) complex tensor
    """
    # Compute base frequencies for each pair dimension
    # (dim // 2,)
    freq_indices = torch.arange(0, dim, 2, device=device).float()
    freqs = 1.0 / (theta ** (freq_indices / dim))

    # Position indices
    # (max_len,)
    positions = torch.arange(max_len, device=device).float()

    # Outer product: position x frequency
    # (max_len,) x (dim // 2,) → (max_len, dim // 2)
    angles = torch.outer(positions, freqs)

    # Convert to complex exponential: e^{i * angle}
    # (max_len, dim // 2) → (max_len, dim // 2) complex
    return torch.polar(torch.ones_like(angles), angles)


def apply_rope(x: torch.Tensor, freqs: torch.Tensor) -> torch.Tensor:
    """Apply RoPE to the last dimension of x.

    x may have shape (..., rope_dim). We treat the last dim as pairs.

    Args:
        x:     (..., rope_dim)  — any leading dims
        freqs: (seq_len, rope_dim // 2) complex
    Returns:
        (..., rope_dim)
    """
    orig_shape = x.shape
    seq_len = x.shape[-3] if x.dim() >= 3 else x.shape[0]

    # Reshape last dim into pairs → view as complex
    # (..., rope_dim) → (..., rope_dim // 2, 2) → (..., rope_dim // 2) complex
    x_pairs = x.float().reshape(*x.shape[:-1], -1, 2)
    x_complex = torch.view_as_complex(x_pairs)

    # Broadcast freqs to match leading dims
    f = freqs[:seq_len]
    shape = [1] * x_complex.dim()
    if x_complex.dim() >= 3:
        shape[-3] = seq_len
    else:
        shape[0] = seq_len
    shape[-1] = f.shape[-1]
    f = f.view(*shape)

    # Rotate via complex multiplication
    x_rotated = x_complex * f

    # Convert back to real
    # (..., rope_dim // 2) complex → (..., rope_dim // 2, 2) → (..., rope_dim)
    return torch.view_as_real(x_rotated).reshape(orig_shape).type_as(x)


## 1.2 Implementation

The full data flow inside MLA:

```
         h_t  (hidden state at position t)
          │
    ┌─────┼──────────────┐
    │     │              │
  W_DKV  W_DQ          W_KR (decoupled RoPE key)
    │     │              │
  c_kv   c_q         k_rope ← RoPE(·)
  (CACHED)│          (CACHED)
    │     │
 ┌──┴──┐  W_UQ
 W_UK  W_UV │
 │     │   q_c
 k_c   v    │
 │          W_QR → q_rope ← RoPE(·)
 │          │
 k=[k_c;k_rope]    q=[q_c;q_rope]
          │
      Attention(q, k, v)
          │
        W_O → output
```

In [ ]:
class MultiHeadLatentAttention(nn.Module):
    """Multi-head Latent Attention (MLA) from DeepSeek-V2.

    Reference: DeepSeek-V2 — A Strong, Economical, and Efficient
               Mixture-of-Experts Language Model (arXiv 2405.04434)

    Key design decisions:
      * Joint KV compression via a single down-projection into c_kv.
        Only c_kv and k_rope need to be cached — dramatically smaller
        than caching full K, V matrices.
      * Decoupled RoPE: position-dependent rotary embeddings are applied
        on a separate, small branch so that the low-rank reconstruction
        of content K remains position-agnostic (enabling the compression
        trick during inference).
      * Optional query compression further reduces activation memory
        during training.
    """

    def __init__(
        self,
        model_dim: int,
        num_heads: int,
        head_dim: int,
        kv_compress_dim: int,
        q_compress_dim: int,
        rope_head_dim: int,
    ):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = head_dim
        self.kv_compress_dim = kv_compress_dim
        self.rope_head_dim = rope_head_dim
        self.attn_dim = head_dim + rope_head_dim

        # --- KV compression path ---
        # Down-project hidden state into compressed KV latent
        # (batch_num, seq_len, model_dim) → (batch_num, seq_len, kv_compress_dim)
        self.w_dkv = nn.Linear(model_dim, kv_compress_dim, bias=False)

        # Up-project latent to content keys (all heads)
        # Input: (batch_num, seq_len, kv_compress_dim)
        # Output: (batch_num, seq_len, num_heads * head_dim)
        self.w_uk = nn.Linear(kv_compress_dim, num_heads * head_dim, bias=False)

        # Up-project latent to values (all heads)
        # Input: (batch_num, seq_len, kv_compress_dim)
        # Output (batch_num, seq_len, num_heads * head_dim)
        self.w_uv = nn.Linear(kv_compress_dim, num_heads * head_dim, bias=False)

        # --- Decoupled RoPE key branch (shared across all heads) ---
        # Input: (batch_num, seq_len, model_dim)
        # Output: (batch_num, seq_len, rope_head_dim)
        self.w_kr = nn.Linear(model_dim, rope_head_dim, bias=False)

        # --- Query compression path ---
        # Input: (batch_num, seq_len, model_dim)
        # Output: (batch_num, seq_len, q_compress_dim)
        self.w_dq = nn.Linear(model_dim, q_compress_dim, bias=False)

        # Up-project to content queries
        # Input: (batch_num, seq_len, q_compress_dim)
        # Output: (batch_num, seq_len, num_heads * head_dim)
        self.w_uq = nn.Linear(q_compress_dim, num_heads * head_dim, bias=False)

        # Decoupled RoPE query branch (per head)
        # Input: (batch_num, seq_len, q_compress_dim)
        # Output: (batch_num, seq_len, num_heads * rope_head_dim)
        self.w_qr = nn.Linear(q_compress_dim, num_heads * rope_head_dim, bias=False)

        # --- Output projection ---
        # Input: (batch_num, seq_len, num_heads * head_dim)
        # Output: (batch_num, seq_len, model_dim)
        self.w_o = nn.Linear(num_heads * head_dim, model_dim, bias=False)

    def forward(
        self,
        x: torch.Tensor,
        rope_freqs: torch.Tensor,
        mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """Forward pass.

        Args:
            x:          (batch_num, seq_len, model_dim)
            rope_freqs: (max_len, rope_head_dim // 2) complex
            mask:       (batch_num, 1, seq_len, seq_len) or None
        Returns:
            output:     (batch_num, seq_len, model_dim)
        """
        batch_num, seq_len, _ = x.shape

        # ── KV compression ──────────────────────────────────────────
        # Compress hidden state into low-rank KV latent
        # (batch_num, seq_len, model_dim) → (batch_num, seq_len, kv_compress_dim)
        c_kv = self.w_dkv(x)

        # Reconstruct content keys from compressed latent
        # (batch_num, seq_len, kv_compress_dim) → (batch_num, seq_len, num_heads, head_dim)
        k_content = self.w_uk(c_kv).view(
            batch_num, seq_len, self.num_heads, self.head_dim
        )

        # Reconstruct values from compressed latent
        # (batch_num, seq_len, kv_compress_dim) → (batch_num, seq_len, num_heads, head_dim)
        v = self.w_uv(c_kv).view(
            batch_num, seq_len, self.num_heads, self.head_dim
        )

        # ── Decoupled RoPE for keys (shared across heads) ──────────
        # (batch_num, seq_len, model_dim) → (batch_num, seq_len, rope_head_dim)
        k_rope = self.w_kr(x)

        # Add a dummy head dim, apply RoPE, then broadcast to all heads
        # (batch_num, seq_len, rope_head_dim) → (batch_num, seq_len, 1, rope_head_dim)
        k_rope = k_rope.unsqueeze(2)
        k_rope = apply_rope(k_rope, rope_freqs)

        # (batch_num, seq_len, 1, rope_head_dim) → (batch_num, seq_len, num_heads, rope_head_dim)
        k_rope = k_rope.expand(-1, -1, self.num_heads, -1)

        # ── Query compression + decoupled RoPE ────────────────────
        # (batch_num, seq_len, model_dim) → (batch_num, seq_len, q_compress_dim)
        c_q = self.w_dq(x)

        # Content queries
        # (batch_num, seq_len, q_compress_dim) → (batch_num, seq_len, num_heads, head_dim)
        q_content = self.w_uq(c_q).view(
            batch_num, seq_len, self.num_heads, self.head_dim
        )

        # RoPE queries (per head)
        # (batch_num, seq_len, q_compress_dim) → (batch_num, seq_len, num_heads, rope_head_dim)
        q_rope = self.w_qr(c_q).view(
            batch_num, seq_len, self.num_heads, self.rope_head_dim
        )
        q_rope = apply_rope(q_rope, rope_freqs)

        # ── Concatenate content + RoPE parts ───────────────────────
        # (batch_num, seq_len, num_heads, head_dim + rope_head_dim)
        q = torch.cat([q_content, q_rope], dim=-1)
        k = torch.cat([k_content, k_rope], dim=-1)

        # ── Scaled dot-product attention ───────────────────────────
        # Transpose to (batch_num, num_heads, seq_len, attn_dim)
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)

        # V stays at head_dim (NOT attn_dim) — the RoPE part only
        # participates in the score, not the value aggregation
        v = v.transpose(1, 2)

        # (batch_num, num_heads, seq_len, attn_dim) @ (batch_num, num_heads, attn_dim, seq_len)
        # → (batch_num, num_heads, seq_len, seq_len)
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.attn_dim)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-inf"))

        # (batch_num, num_heads, seq_len, seq_len)
        attn_weights = F.softmax(scores, dim=-1)

        # (batch_num, num_heads, seq_len, seq_len) @ (batch_num, num_heads, seq_len, head_dim)
        # → (batch_num, num_heads, seq_len, head_dim)
        context = torch.matmul(attn_weights, v)

        # Merge heads
        # (batch_num, num_heads, seq_len, head_dim) → (batch_num, seq_len, num_heads * head_dim)
        context = context.transpose(1, 2).contiguous().view(
            batch_num, seq_len, self.num_heads * self.head_dim
        )

        # Final projection
        # (batch_num, seq_len, num_heads * head_dim) → (batch_num, seq_len, model_dim)
        return self.w_o(context)

In [ ]:
# ── Test MLA and compare cache sizes with standard MHA / GQA ──────
model_dim = 2048
num_heads = 16
head_dim = 128
kv_compress_dim = 512
q_compress_dim = 1536
rope_head_dim = 64
max_len = 128

mla = MultiHeadLatentAttention(
    model_dim=model_dim,
    num_heads=num_heads,
    head_dim=head_dim,
    kv_compress_dim=kv_compress_dim,
    q_compress_dim=q_compress_dim,
    rope_head_dim=rope_head_dim,
).to(device)

rope_freqs = precompute_rope_frequencies(rope_head_dim, max_len, device=device)

# Synthetic input: (batch_num, seq_len, model_dim)
x = torch.randn(2, 64, model_dim, device=device)

# Causal mask: (1, 1, seq_len, seq_len)
causal_mask = torch.tril(torch.ones(64, 64, device=device)).unsqueeze(0).unsqueeze(0)

out = mla(x, rope_freqs, mask=causal_mask)
print(f"MLA output shape: {out.shape}")  # (2, 64, 2048)

# ── KV cache size comparison ──────────────────────────────────────
mha_cache_per_token = 2 * num_heads * head_dim
gqa_kv_heads = 4
gqa_cache_per_token = 2 * gqa_kv_heads * head_dim
mla_cache_per_token = kv_compress_dim + rope_head_dim

print(f"\nKV cache elements per token:")
print(f"  Standard MHA : {mha_cache_per_token:>6}")
print(f"  GQA (4 heads): {gqa_cache_per_token:>6}")
print(f"  MLA          : {mla_cache_per_token:>6}")
print(f"  MLA reduction vs MHA: {mha_cache_per_token / mla_cache_per_token:.1f}×")

MLA output shape: torch.Size([2, 64, 2048])

KV cache elements per token:
  Standard MHA :   4096
  GQA (4 heads):   1024
  MLA          :    576
  MLA reduction vs MHA: 7.1×


---
# 2) Mixture of Experts (MoE)

Standard MoE (e.g., Switch Transformer) uses a **small number of large experts**.
DeepSeekMoE instead uses **many small experts**, achieving finer-grained
specialization and better parameter utilisation.

| Feature | Standard MoE | DeepSeekMoE |
|---|---|---|
| Expert count | 8–16 large | 64+ fine-grained |
| Always-on experts | None | K_s shared experts |
| Load balancing | Auxiliary loss | **Bias-based** (loss-free in V3) |

### Shared Expert Isolation

Some knowledge (e.g., syntax, common phrases) is needed by **every** token.
Dedicating a few experts as *shared* (always activated) captures this
common knowledge, freeing the routed experts to specialise.

### Auxiliary-Loss-Free Load Balancing (DeepSeek-V3)

Traditional MoE adds an auxiliary loss to encourage balanced routing.
DeepSeek-V3 instead adds a learnable **bias** to the routing logits for
expert selection only — the bias does **not** affect the softmax weights
used to combine expert outputs. This avoids the auxiliary loss's tendency
to degrade model quality.

In [ ]:
class ExpertFFN(nn.Module):
    """Single fine-grained expert: a SwiGLU feed-forward network.

    Compared to standard Transformer FFN (4 × model_dim intermediate),
    each fine-grained expert uses a much smaller intermediate dimension
    (expert_dim), since many experts are activated in parallel.
    """

    def __init__(self, model_dim: int, expert_dim: int):
        super().__init__()
        # Gate projection for SwiGLU
        # (batch_num, seq_len, model_dim) → (batch_num, seq_len, expert_dim)
        self.w_gate = nn.Linear(model_dim, expert_dim, bias=False)

        # Up projection for SwiGLU
        # (batch_num, seq_len, model_dim) → (batch_num, seq_len, expert_dim)
        self.w_up = nn.Linear(model_dim, expert_dim, bias=False)

        # Down projection back to model dimension
        # (batch_num, seq_len, expert_dim) → (batch_num, seq_len, model_dim)
        self.w_down = nn.Linear(expert_dim, model_dim, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # SwiGLU: silu(gate) * up → down
        # Input: (batch_num, num_tokens, model_dim)
        # Output: (batch_num, num_tokens, model_dim)
        return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))

In [ ]:
class TopKRouter(nn.Module):
    """Token-level top-K router with auxiliary-loss-free load balancing.

    Design decisions:
      * A bias vector is added to routing logits for expert SELECTION only.
      * The softmax weights used to COMBINE expert outputs are computed
        from the ORIGINAL (unbiased) logits. This prevents the bias from
        distorting the learned representations.
      * The bias is updated via a simple heuristic (not gradient descent):
        increase bias for underloaded experts, decrease for overloaded.
    """

    def __init__(self, model_dim: int, num_experts: int, top_k: int):
        super().__init__()
        self.top_k = top_k
        self.num_experts = num_experts

        # Routing projection: token → expert scores
        # (batch_num * seq_len, model_dim) → (batch_num * seq_len, num_experts)
        self.gate = nn.Linear(model_dim, num_experts, bias=False)

        # Auxiliary-loss-free bias (NOT trained by gradient; updated heuristically)
        self.expert_bias = nn.Parameter(
            torch.zeros(num_experts), requires_grad=False
        )

    def forward(
        self, x: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """Route tokens to top-K experts.

        Args:
            x: (num_tokens, model_dim)
        Returns:
            weights: (num_tokens, top_k)  — normalised expert weights
            indices: (num_tokens, top_k)  — selected expert indices
        """
        # Compute raw routing logits
        # (num_tokens, model_dim) → (num_tokens, num_experts)
        logits = self.gate(x)

        # Biased logits for SELECTION (bias helps balance load)
        # (num_tokens, num_experts)
        routing_logits = logits + self.expert_bias

        # Select top-K experts per token
        # (num_tokens, num_experts) → (num_tokens, top_k)
        _, indices = torch.topk(routing_logits, self.top_k, dim=-1)

        # Compute combination weights from ORIGINAL (unbiased) logits
        # Gather original logits at selected positions
        # (num_tokens, top_k)
        selected_logits = logits.gather(-1, indices)

        # Normalise via softmax over selected experts
        # (num_tokens, top_k)
        weights = F.softmax(selected_logits, dim=-1)

        return weights, indices

In [ ]:
class DeepSeekMoE(nn.Module):
    """DeepSeek Mixture-of-Experts layer.

    Three key components:
      1. Shared experts — always activated for common knowledge.
      2. Routed (fine-grained) experts — dynamically selected per token.
      3. Top-K router with auxiliary-loss-free bias balancing.
    """

    def __init__(
        self,
        model_dim: int,
        num_shared_experts: int,
        num_routed_experts: int,
        num_active_experts: int,
        expert_dim: int,
    ):
        super().__init__()
        self.num_shared = num_shared_experts
        self.num_routed = num_routed_experts
        self.num_active = num_active_experts

        # Shared experts: always contribute to every token
        self.shared_experts = nn.ModuleList(
            [ExpertFFN(model_dim, expert_dim) for _ in range(num_shared_experts)]
        )

        # Routed (fine-grained) experts: dynamically selected
        self.routed_experts = nn.ModuleList(
            [ExpertFFN(model_dim, expert_dim) for _ in range(num_routed_experts)]
        )

        # Token-level router
        self.router = TopKRouter(model_dim, num_routed_experts, num_active_experts)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass.

        Args:
            x: (batch_num, seq_len, model_dim)
        Returns:
            output: (batch_num, seq_len, model_dim)
        """
        batch_num, seq_len, model_dim = x.shape

        # ── Shared experts (always activated) ─────────────────────
        # Sum outputs from all shared experts
        # (batch_num, seq_len, model_dim)
        shared_output = sum(expert(x) for expert in self.shared_experts)

        # ── Routed experts ────────────────────────────────────────
        # Flatten batch and sequence dims for routing
        # (batch_num, seq_len, model_dim) → (num_tokens, model_dim)
        num_tokens = batch_num * seq_len
        flat_x = x.view(num_tokens, model_dim)

        # Get top-K expert assignments
        # weights: (num_tokens, num_active_experts)
        # indices: (num_tokens, num_active_experts)
        weights, indices = self.router(flat_x)

        # Accumulate weighted expert outputs
        # (num_tokens, model_dim)
        routed_output = torch.zeros_like(flat_x)

        # Iterate over each active expert slot
        for k in range(self.num_active):
            # Expert indices for slot k: (num_tokens,)
            expert_ids = indices[:, k]
            # Corresponding weights: (num_tokens, 1)
            expert_weights = weights[:, k].unsqueeze(-1)

            # Process tokens assigned to each expert
            for e_idx in range(self.num_routed):
                # Boolean mask of tokens routed to expert e_idx in slot k
                token_mask = expert_ids == e_idx
                if not token_mask.any():
                    continue

                # Gather tokens for this expert
                # (num_selected, model_dim)
                expert_input = flat_x[token_mask]

                # Run expert FFN
                # (num_selected, model_dim) → (num_selected, model_dim)
                expert_out = self.routed_experts[e_idx](expert_input)

                # Weighted addition
                routed_output[token_mask] += expert_weights[token_mask] * expert_out

        # Reshape back to (batch_num, seq_len, model_dim)
        routed_output = routed_output.view(batch_num, seq_len, model_dim)

        # Combine shared + routed outputs
        return shared_output + routed_output

In [ ]:
# ── Test DeepSeekMoE ──────────────────────────────────────────────
moe = DeepSeekMoE(
    model_dim=2048,
    num_shared_experts=2,
    num_routed_experts=16,
    num_active_experts=4,
    expert_dim=1024,
).to(device)

x = torch.randn(2, 64, 2048, device=device)
out = moe(x)
print(f"MoE output shape: {out.shape}")  # (2, 64, 2048)

total_params = sum(p.numel() for p in moe.parameters())
print(f"Total MoE parameters: {total_params / 1e6:.1f}M")

# Per forward pass, only shared + top-K routed experts are activated
shared_params = sum(p.numel() for e in moe.shared_experts for p in e.parameters())
per_expert_params = sum(p.numel() for p in moe.routed_experts[0].parameters())
active_params = shared_params + 4 * per_expert_params
print(f"Activated parameters per token: {active_params / 1e6:.1f}M / {total_params / 1e6:.1f}M")

MoE output shape: torch.Size([2, 64, 2048])
Total MoE parameters: 113.3M
Activated parameters per token: 37.7M / 113.3M


# 3) Multi-Token Prediction (MTP)

Standard causal LMs predict **one** next token per position. MTP
(introduced in DeepSeek-V3) adds extra prediction heads that forecast
**multiple** future tokens at each position.

During training this provides a richer gradient signal; during inference
the extra heads enable speculative decoding for faster generation.

### Data flow

```
Hidden state h_t  ──→  Head 0: predict token t+1  (standard LM head)
                  ──→  Head 1: predict token t+2
                  ──→  Head k: predict token t+k+1
```

Each extra head uses a **separate** small Transformer layer that takes the
concatenation of the current hidden state and the *previous head's*
embedding prediction, forming a sequential chain.

In [ ]:
class MTPHead(nn.Module):
    """Single Multi-Token Prediction head.

    Each MTP head receives the previous head's hidden state (or the main
    Transformer output for head 0), concatenated with the embedding of the
    predicted token from the previous depth, then processes it through a
    small Transformer layer to predict a token further into the future.
    """

    def __init__(self, model_dim: int, vocab_size: int, num_heads: int = 4):
        super().__init__()
        # Project concatenated [hidden_state; prev_embedding] back to model_dim
        # (batch_num, seq_len, 2 * model_dim) → (batch_num, seq_len, model_dim)
        self.proj = nn.Linear(2 * model_dim, model_dim, bias=False)

        # Lightweight self-attention layer for this prediction depth
        self.norm = nn.RMSNorm(model_dim)
        self.attn = nn.MultiheadAttention(
            model_dim, num_heads, batch_first=True
        )

        # Vocabulary projection
        # (batch_num, seq_len, model_dim) → (batch_num, seq_len, vocab_size)
        self.head = nn.Linear(model_dim, vocab_size, bias=False)

    def forward(
        self,
        hidden: torch.Tensor,
        prev_embed: torch.Tensor,
        causal_mask: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """Forward pass for one MTP depth.

        Args:
            hidden:      (batch_num, seq_len, model_dim)
            prev_embed:  (batch_num, seq_len, model_dim)
            causal_mask: (seq_len, seq_len)
        Returns:
            logits:      (batch_num, seq_len, vocab_size)
            new_hidden:  (batch_num, seq_len, model_dim)
        """
        # Concatenate hidden state with previous depth's embedding
        # (batch_num, seq_len, 2 * model_dim)
        combined = torch.cat([hidden, prev_embed], dim=-1)

        # Project back to model_dim
        # (batch_num, seq_len, model_dim)
        h = self.proj(combined)

        # Self-attention with causal mask
        h_norm = self.norm(h)
        # (batch_num, seq_len, model_dim)
        attn_out, _ = self.attn(
            h_norm, h_norm, h_norm, attn_mask=causal_mask, is_causal=True
        )
        new_hidden = h + attn_out

        # Predict token at this depth
        # (batch_num, seq_len, vocab_size)
        logits = self.head(self.norm(new_hidden))

        return logits, new_hidden


class MultiTokenPredictor(nn.Module):
    """Multi-Token Prediction module with D extra prediction depths."""

    def __init__(
        self,
        model_dim: int,
        vocab_size: int,
        num_depths: int = 2,
        num_heads: int = 4,
    ):
        super().__init__()
        self.num_depths = num_depths

        # Shared embedding (same as main model)
        self.embed = nn.Embedding(vocab_size, model_dim)

        # Main LM head (depth 0): predict next token
        self.main_head = nn.Linear(model_dim, vocab_size, bias=False)

        # Extra MTP heads (depths 1..D): predict tokens further ahead
        self.mtp_heads = nn.ModuleList(
            [MTPHead(model_dim, vocab_size, num_heads) for _ in range(num_depths)]
        )

    def forward(
        self,
        hidden: torch.Tensor,
        target_ids: torch.Tensor,
    ) -> torch.Tensor:
        """Compute combined MTP loss.

        Args:
            hidden:     (batch_num, seq_len, model_dim) — Transformer output
            target_ids: (batch_num, seq_len) — ground-truth token IDs
        Returns:
            loss: scalar — weighted sum of per-depth cross-entropy losses
        """
        batch_num, seq_len, model_dim = hidden.shape

        # Causal mask for internal attention in MTP heads
        causal_mask = torch.triu(
            torch.full((seq_len, seq_len), float("-inf"), device=hidden.device),
            diagonal=1,
        )

        # ── Depth 0: standard next-token prediction ───────────────
        # (batch_num, seq_len, vocab_size)
        logits_0 = self.main_head(hidden)

        # Target for depth 0: token at position t+1
        # (batch_num, seq_len - 1)
        loss_0 = F.cross_entropy(
            logits_0[:, :-1].reshape(-1, logits_0.size(-1)),
            target_ids[:, 1:].reshape(-1),
            ignore_index=0,
        )

        total_loss = loss_0

        # ── Depths: predict further future tokens ────────────
        current_hidden = hidden
        # Embedding of ground-truth next token as input to depth 1
        prev_embed = self.embed(target_ids)

        for d, mtp_head in enumerate(self.mtp_heads):
            depth = d + 1

            # MTP head produces logits and updated hidden state
            # (batch_num, seq_len, vocab_size), (batch_num, seq_len, model_dim)
            logits_d, current_hidden = mtp_head(
                current_hidden, prev_embed, causal_mask
            )

            # Target for depth d: token at position t + d + 1
            shift = depth + 1
            if shift >= seq_len:
                break

            # (batch_num, seq_len - shift)
            loss_d = F.cross_entropy(
                logits_d[:, : -shift].reshape(-1, logits_d.size(-1)),
                target_ids[:, shift:].reshape(-1),
                ignore_index=0,
            )

            # Weight deeper predictions less (simple linear decay)
            weight = 1.0 / (depth + 1)
            total_loss = total_loss + weight * loss_d

            # Prepare prev_embed for next depth using ground-truth token
            if depth < self.num_depths:
                prev_embed = self.embed(
                    target_ids.roll(-depth, dims=1)
                )

        return total_loss

In [ ]:
# ── Test Multi-Token Prediction ───────────────────────────────────

vocab_size_mtp = 1000
model_dim_mtp = 256

mtp = MultiTokenPredictor(
    model_dim=model_dim_mtp,
    vocab_size=vocab_size_mtp,
    num_depths=2,
    num_heads=4,
).to(device)

# Synthetic hidden states and target IDs
hidden = torch.randn(2, 32, model_dim_mtp, device=device)
target_ids = torch.randint(1, vocab_size_mtp, (2, 32), device=device)

loss = mtp(hidden, target_ids)
print(f"MTP combined loss: {loss.item():.4f}")

MTP combined loss: 12.9822


---
# 4) DeepSeek Transformer Block

A single DeepSeek-V2/V3 decoder block combines:

1. **RMSNorm** → **MLA** → residual
2. **RMSNorm** → **DeepSeekMoE** → residual

This is the same pre-norm Transformer pattern as LLaMA, but with
MLA replacing GQA and MoE replacing the dense FFN.

In [ ]:
class DeepSeekBlock(nn.Module):
    """Single DeepSeek decoder block: MLA + MoE with pre-norm residuals."""

    def __init__(self, config: dict):
        super().__init__()

        model_dim = config["model_dim"]

        # Pre-norm for attention
        self.attn_norm = nn.RMSNorm(model_dim)

        # Multi-head Latent Attention
        self.attn = MultiHeadLatentAttention(
            model_dim=model_dim,
            num_heads=config["num_heads"],
            head_dim=config["head_dim"],
            kv_compress_dim=config["kv_compress_dim"],
            q_compress_dim=config["q_compress_dim"],
            rope_head_dim=config["rope_head_dim"],
        )

        # Pre-norm for MoE
        self.moe_norm = nn.RMSNorm(model_dim)

        # DeepSeek MoE feed-forward
        self.moe = DeepSeekMoE(
            model_dim=model_dim,
            num_shared_experts=config["num_shared_experts"],
            num_routed_experts=config["num_routed_experts"],
            num_active_experts=config["num_active_experts"],
            expert_dim=config["expert_dim"],
        )

    def forward(
        self,
        x: torch.Tensor,
        rope_freqs: torch.Tensor,
        mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """Forward pass.

        Args:
            x:          (batch_num, seq_len, model_dim)
            rope_freqs: (max_len, rope_head_dim // 2) complex
            mask:       (batch_num, 1, seq_len, seq_len)
        Returns:
            output:     (batch_num, seq_len, model_dim)
        """
        # Pre-norm → MLA → residual
        # (batch_num, seq_len, model_dim)
        h = x + self.attn(self.attn_norm(x), rope_freqs, mask)

        # Pre-norm → MoE → residual
        # (batch_num, seq_len, model_dim)
        out = h + self.moe(self.moe_norm(h))

        return out


# ── Test full DeepSeek block ──────────────────────────────────────
config = dict(
    model_dim=2048,
    num_heads=16,
    head_dim=128,
    kv_compress_dim=512,
    q_compress_dim=1536,
    rope_head_dim=64,
    num_shared_experts=2,
    num_routed_experts=16,
    num_active_experts=4,
    expert_dim=1024,
)

block = DeepSeekBlock(config).to(device)
rope_freqs = precompute_rope_frequencies(config["rope_head_dim"], 256, device=device)
x = torch.randn(2, 64, 2048, device=device)
causal_mask = torch.tril(torch.ones(64, 64, device=device)).unsqueeze(0).unsqueeze(0)

out = block(x, rope_freqs, causal_mask)
print(f"DeepSeek block output: {out.shape}")  # (2, 64, 2048)
print(f"Total block parameters: {sum(p.numel() for p in block.parameters()) / 1e6:.1f}M")

DeepSeek block output: torch.Size([2, 64, 2048])
Total block parameters: 128.6M
